# Modelamiento y Arquetipos — Estudio PENSER Egresados USTA
**Proyecto:** Consultorio de Estadística y Ciencia de Datos  
**Equipo:** Yeimy Alarcón · Karen Suarez · Maria José Galindo  
**Semestre:** 2025-2

Este notebook presenta el proceso de reducción dimensional (PCA) y la identificación
de arquetipos de graduados mediante clustering. Los resultados se basan en la base
procesada generada por el pipeline `ingest → features → train → evaluate`.

---

## 1. Configuración e Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

COLORES_ARQUETIPOS = ['#F96167', '#065A82', '#2C5F2D']
NOMBRES_ARQUETIPOS = {
    0: 'El Subjetivamente Satisfecho',
    1: 'El Profesional Consolidado',
    2: 'El Líder de Alto Desempeño'
}

print('Librerías cargadas correctamente.')

## 2. Carga de Datos

In [ ]:
# Base procesada
df = pd.read_parquet('data/processed/base_procesada.parquet')

# Base con arquetipos (generada por train.py)
df_arq = pd.read_parquet('artifacts/base_con_arquetipos.parquet')

print(f'Base procesada    : {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Base con arquetipos: {df_arq.shape[0]:,} filas × {df_arq.shape[1]} columnas')
print(f'Arquetipos encontrados: {sorted(df_arq["arquetipo"].unique())}')

## 3. Preparación de Variables para el Modelamiento

In [ ]:
COLS_COMPETENCIAS = [
    'com_escrita', 'com_oral', 'pensamiento_critico', 'metodos_cuantitativos',
    'metodos_cualitativos', 'lectura_academica', 'argumentacion', 'segunda_lengua',
    'creatividad', 'resolucion_conflictos', 'liderazgo', 'toma_decisiones',
    'resolucion_problemas', 'investigacion', 'herramientas_informaticas',
    'contextos_multiculturales', 'insercion_laboral', 'herramientas_modernas',
    'gestion_informacion', 'trabajo_equipo', 'aprendizaje_autonomo',
    'conocimientos_multidisciplinares', 'etica'
]
COLS_BIENESTAR = [
    'adquirio_bienes', 'mejoro_vivienda', 'mejoro_salud',
    'acceso_seguridad_social', 'incremento_cultural', 'satisfecho_ocio', 'red_amigos'
]
COLS_ADICIONALES = [
    'movilidad_social', 'satisfaccion_vida', 'satisfaccion_formacion',
    'efecto_calidad_vida', 'score_bienestar'
]

todas = COLS_COMPETENCIAS + COLS_BIENESTAR + COLS_ADICIONALES
cols_disponibles = [c for c in todas if c in df.columns]

X = df[cols_disponibles].copy()
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce')

# Imputar y estandarizar
imputer = SimpleImputer(strategy='median')
X_imp = imputer.fit_transform(X)
X_scaled = StandardScaler().fit_transform(X_imp)

print(f'Variables para modelamiento: {len(cols_disponibles)}')
print(f'Registros completos (tras imputación): {X_scaled.shape[0]:,}')

## 4. Análisis de Componentes Principales (PCA)

In [ ]:
# PCA completo para análisis de varianza
pca_full = PCA()
pca_full.fit(X_scaled)

var_individual = pca_full.explained_variance_ratio_ * 100
var_acumulada = np.cumsum(var_individual)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Varianza individual
n_show = 15
axes[0].bar(range(1, n_show+1), var_individual[:n_show], color='#065A82', alpha=0.8, edgecolor='white')
axes[0].axhline(y=1/len(cols_disponibles)*100, color='red', linestyle='--', 
                alpha=0.7, label=f'Criterio Kaiser ({1/len(cols_disponibles)*100:.1f}%)')
axes[0].set_title('Varianza Explicada por Componente', fontweight='bold')
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Varianza Explicada (%)')
axes[0].legend()

# Varianza acumulada
axes[1].plot(range(1, len(var_acumulada)+1), var_acumulada, 'o-', color='#2C5F2D', linewidth=2)
axes[1].axhline(y=85, color='orange', linestyle='--', alpha=0.7, label='Umbral 85%')
axes[1].axhline(y=95, color='red', linestyle='--', alpha=0.7, label='Umbral 95%')
n_85 = int(np.argmax(var_acumulada >= 85) + 1)
axes[1].axvline(x=n_85, color='orange', linestyle=':', alpha=0.7)
axes[1].set_title('Varianza Acumulada', fontweight='bold')
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('Varianza Acumulada (%)')
axes[1].legend()
axes[1].set_xlim(1, 20)

plt.suptitle('Análisis de Componentes Principales — PENSER USTA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Componentes para 85% de varianza: {n_85}')
print(f'PC1 explica: {var_individual[0]:.1f}% de la varianza')
print(f'PC2 explica: {var_individual[1]:.1f}% de la varianza')

## 5. Selección del Número de Clusters

In [ ]:
# PCA reducido para clustering
pca = PCA(n_components=n_85)
X_pca = pca.fit_transform(X_scaled)

# Métricas para diferentes k
k_range = range(2, 8)
siluetas_km = []
inercias = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    siluetas_km.append(silhouette_score(X_pca, labels))
    inercias.append(km.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Método del codo
axes[0].plot(list(k_range), inercias, 'o-', color='#065A82', linewidth=2, markersize=8)
axes[0].set_title('Método del Codo (Inercia)', fontweight='bold')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inercia')
axes[0].set_xticks(list(k_range))

# Coeficiente de silueta
colores_sil = ['#F96167' if s == max(siluetas_km) else '#97BC62' for s in siluetas_km]
axes[1].bar(list(k_range), siluetas_km, color=colores_sil, edgecolor='white')
axes[1].set_title('Coeficiente de Silueta por k', fontweight='bold')
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silueta (mayor = mejor)')
axes[1].set_xticks(list(k_range))
k_optimo = list(k_range)[siluetas_km.index(max(siluetas_km))]
axes[1].axvline(x=k_optimo, color='red', linestyle='--', alpha=0.7, label=f'k óptimo = {k_optimo}')
axes[1].legend()
for i, (k, s) in enumerate(zip(k_range, siluetas_km)):
    axes[1].text(k, s + 0.002, f'{s:.3f}', ha='center', fontsize=9)

plt.suptitle('Selección del Número Óptimo de Clusters', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'k óptimo por silueta (k≥3): {k_optimo}')

## 6. Dendrograma — Clustering Jerárquico

In [ ]:
# Muestra aleatoria para el dendrograma (máx 200 registros)
np.random.seed(42)
idx = np.random.choice(len(X_pca), min(200, len(X_pca)), replace=False)
X_sample = X_pca[idx]

Z = linkage(X_sample, method='ward')

fig, ax = plt.subplots(figsize=(16, 6))
dendrogram(
    Z, ax=ax,
    truncate_mode='lastp', p=30,
    leaf_rotation=90, leaf_font_size=8,
    color_threshold=Z[-3, 2],
    above_threshold_color='gray'
)
ax.set_title('Dendrograma — Clustering Jerárquico Ward (muestra n=200)', fontweight='bold', fontsize=13)
ax.set_xlabel('Índice del graduado')
ax.set_ylabel('Distancia (Ward)')
ax.axhline(y=Z[-3, 2], color='red', linestyle='--', alpha=0.7, label=f'Corte para k=3')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Distribución de Arquetipos

In [ ]:
arquetipos = df_arq['arquetipo'].astype(int)
dist = arquetipos.value_counts().sort_index()
nombres = [f'{i}\n{NOMBRES_ARQUETIPOS.get(i, "")}' for i in dist.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Barras
bars = axes[0].bar(range(len(dist)), dist.values, color=COLORES_ARQUETIPOS, edgecolor='white', width=0.6)
axes[0].set_title('Distribución de Arquetipos', fontweight='bold')
axes[0].set_xticks(range(len(dist)))
axes[0].set_xticklabels(nombres, fontsize=9)
axes[0].set_ylabel('Número de graduados')
for bar, v in zip(bars, dist.values):
    pct = v / len(arquetipos) * 100
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 5, 
                 f'{v}\n({pct:.1f}%)', ha='center', fontsize=9, fontweight='bold')

# Pastel
wedges, texts, autotexts = axes[1].pie(
    dist.values,
    labels=[NOMBRES_ARQUETIPOS.get(i, f'Arquetipo {i}') for i in dist.index],
    colors=COLORES_ARQUETIPOS,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for text in texts:
    text.set_fontsize(9)
axes[1].set_title('Proporción de Arquetipos', fontweight='bold')

plt.suptitle(f'Arquetipos de Graduados USTA (n={len(arquetipos):,})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Visualización PCA — Separación de Arquetipos

In [ ]:
# PCA 2D para visualización
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)

labels_arq = df_arq['arquetipo'].astype(int).values

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter por arquetipo
for arq in sorted(np.unique(labels_arq)):
    mask = labels_arq == arq
    axes[0].scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        c=COLORES_ARQUETIPOS[arq],
        label=NOMBRES_ARQUETIPOS.get(arq, f'Arquetipo {arq}'),
        alpha=0.5, s=20, edgecolors='none'
    )
axes[0].set_title('Arquetipos en Espacio PCA (2D)', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend(fontsize=8, loc='upper right')

# Centroides
for arq in sorted(np.unique(labels_arq)):
    mask = labels_arq == arq
    cx, cy = X_2d[mask, 0].mean(), X_2d[mask, 1].mean()
    axes[0].scatter(cx, cy, c=COLORES_ARQUETIPOS[arq], s=200, marker='*', edgecolors='black', linewidth=1, zorder=5)

# Densidad por arquetipo
for arq in sorted(np.unique(labels_arq)):
    mask = labels_arq == arq
    sns.kdeplot(
        x=X_2d[mask, 0],
        ax=axes[1],
        color=COLORES_ARQUETIPOS[arq],
        label=NOMBRES_ARQUETIPOS.get(arq, f'Arquetipo {arq}'),
        fill=True, alpha=0.3, linewidth=2
    )
axes[1].set_title('Densidad por Arquetipo (PC1)', fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel('Densidad')
axes[1].legend(fontsize=8)

plt.suptitle('Separación de Arquetipos en Espacio Reducido (PCA)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Perfil de Competencias por Arquetipo

In [ ]:
cols_comp = [c for c in COLS_COMPETENCIAS if c in df_arq.columns]
df_arq_num = df_arq.copy()
for col in cols_comp:
    df_arq_num[col] = pd.to_numeric(df_arq_num[col], errors='coerce')

perfil = df_arq_num.groupby('arquetipo')[cols_comp].mean()

fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(cols_comp))
width = 0.25

for i, arq in enumerate(sorted(perfil.index)):
    bars = ax.bar(
        x + i * width,
        perfil.loc[arq],
        width, label=NOMBRES_ARQUETIPOS.get(arq, f'Arquetipo {arq}'),
        color=COLORES_ARQUETIPOS[arq], alpha=0.85, edgecolor='white'
    )

ax.set_title('Perfil de Competencias por Arquetipo', fontweight='bold', fontsize=13)
ax.set_xlabel('Competencia')
ax.set_ylabel('Media (escala 1–5)')
ax.set_xticks(x + width)
ax.set_xticklabels(cols_comp, rotation=45, ha='right', fontsize=8)
ax.set_ylim(0, 5.5)
ax.axhline(y=3.0, color='gray', linestyle=':', alpha=0.5)
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

## 10. Radar Chart — Perfil de Cada Arquetipo

In [ ]:
# Seleccionar 8 competencias clave para el radar
comp_radar = [
    'segunda_lengua', 'insercion_laboral', 'herramientas_modernas',
    'liderazgo', 'toma_decisiones', 'trabajo_equipo', 'etica', 'pensamiento_critico'
]
comp_radar = [c for c in comp_radar if c in df_arq.columns]

N = len(comp_radar)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw=dict(polar=True))

for ax, arq in zip(axes, sorted(perfil.index)):
    vals = [perfil.loc[arq, c] for c in comp_radar]
    vals += vals[:1]
    
    ax.plot(angles, vals, 'o-', linewidth=2, color=COLORES_ARQUETIPOS[arq])
    ax.fill(angles, vals, alpha=0.25, color=COLORES_ARQUETIPOS[arq])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(comp_radar, size=8)
    ax.set_ylim(0, 5)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_yticklabels(['1', '2', '3', '4', '5'], size=7)
    ax.set_title(f'Arquetipo {arq}\n{NOMBRES_ARQUETIPOS.get(arq, "")}', 
                 fontweight='bold', pad=15, fontsize=10, color=COLORES_ARQUETIPOS[arq])
    ax.grid(True, alpha=0.3)

plt.suptitle('Radar de Competencias Clave por Arquetipo', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 11. Bienestar y Movilidad Social por Arquetipo

In [ ]:
cols_bien = [c for c in COLS_BIENESTAR if c in df_arq.columns]
df_arq_num2 = df_arq.copy()
for col in cols_bien + ['movilidad_social', 'score_bienestar']:
    if col in df_arq_num2.columns:
        df_arq_num2[col] = pd.to_numeric(df_arq_num2[col], errors='coerce')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bienestar por arquetipo
bien_arq = df_arq_num2.groupby('arquetipo')[cols_bien].mean() * 100
bien_arq.T.plot(kind='bar', ax=axes[0], color=COLORES_ARQUETIPOS, edgecolor='white', width=0.7)
axes[0].set_title('Indicadores de Bienestar por Arquetipo (%)', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('% que respondió Sí')
axes[0].tick_params(axis='x', rotation=30)
axes[0].axhline(y=50, color='gray', linestyle='--', alpha=0.5)
axes[0].legend([NOMBRES_ARQUETIPOS.get(i, f'Arq {i}') for i in bien_arq.index], 
               fontsize=8, loc='lower right')

# Score bienestar y movilidad social
if 'score_bienestar' in df_arq_num2.columns:
    score_arq = df_arq_num2.groupby('arquetipo')['score_bienestar'].mean()
    bars = axes[1].bar(
        [NOMBRES_ARQUETIPOS.get(i, f'Arq {i}') for i in score_arq.index],
        score_arq.values,
        color=COLORES_ARQUETIPOS, edgecolor='white'
    )
    axes[1].set_title('Score de Bienestar Promedio por Arquetipo', fontweight='bold')
    axes[1].set_ylabel('Score promedio (0–7)')
    axes[1].set_ylim(0, 8)
    axes[1].tick_params(axis='x', rotation=20)
    axes[1].axhline(y=score_arq.mean(), color='gray', linestyle='--', 
                    alpha=0.7, label=f'Media global: {score_arq.mean():.2f}')
    axes[1].legend()
    for bar, v in zip(bars, score_arq.values):
        axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.05, 
                     f'{v:.2f}', ha='center', fontweight='bold')

plt.suptitle('Bienestar por Arquetipo — Impacto de la Formación USTA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Resumen de Resultados

In [ ]:
print('=' * 65)
print('  RESUMEN DE ARQUETIPOS — ESTUDIO PENSER EGRESADOS USTA')
print('=' * 65)

for arq in sorted(df_arq['arquetipo'].astype(int).unique()):
    sub = df_arq_num[df_arq_num['arquetipo'].astype(int) == arq]
    n = len(sub)
    pct = n / len(df_arq) * 100
    medias_arq = sub[cols_comp].mean().sort_values()
    print(f"""
  Arquetipo {arq} — {NOMBRES_ARQUETIPOS.get(arq, '')} (n={n}, {pct:.1f}%)
  {'─' * 55}
  ✅ Fortalezas : {', '.join(medias_arq.tail(3).index.tolist())}
  ⚠️  Brechas    : {', '.join(medias_arq.head(3).index.tolist())}
  Media competencias: {medias_arq.mean():.2f} / 5.0""")

print(f"""
{'=' * 65}
  HALLAZGOS TRANSVERSALES
{'=' * 65}

  1. Segunda lengua es la brecha más crítica en los 3 arquetipos
  2. El 49% son Profesionales Consolidados — impacto positivo general
  3. El Líder de Alto Desempeño tiene score bienestar 4.65 / 7
  4. La movilidad social es positiva (~21% ascendió de estrato)
{'=' * 65}
""")